# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This is a binary classification and ranking problem: the model estimates whether a content page will experience an impressions decline greater than 20% in the following month. I will use each classifier’s predicted probability to rank pages for human refresh review.

I will begin with Logistic Regression because it is fast, interpretable, and provides a strong simple comparison with my rule-based baseline. I will then train a Random Forest as a nonlinear challenger because ranking decline risk may depend on interactions among impression momentum, search position, CTR, and traffic volatility. I will select the more complex model only if it produces a meaningful improvement on the same held-out data and metrics.

My primary metric is Precision@20 because the output is a limited-capacity review queue. I will also report Precision@50, Average Precision, ROC AUC, and the evaluation-set base rate. Both models and the Week-4 baseline will be evaluated on identical rows.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)


Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


In [4]:
MONTHS = [
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

monthly_queries = []

for month in MONTHS:
    monthly_queries.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            '{month}' AS month_key,

            COUNT(*) AS observed_days,

            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position,

            AVG(gsc_impressions) AS daily_impression_mean,
            STDDEV_SAMP(gsc_impressions) AS daily_impression_std,

            STDDEV_SAMP(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS daily_position_std,

            SUM(
                CASE
                    WHEN gsc_impressions > 0 THEN 1
                    ELSE 0
                END
            ) AS days_with_impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/'
            'month={month}/*.parquet'
        )

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    """)

monthly_union_sql = "\nUNION ALL\n".join(monthly_queries)

monthly_summary = con.sql(monthly_union_sql).df()

monthly_summary = monthly_summary.sort_values(
    ["month_key", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

print(f"Monthly summary rows: {len(monthly_summary):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly summary rows: 971,603


In [5]:
monthly_coverage = (
    monthly_summary
    .groupby("month_key")
    .agg(
        page_rows=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        minimum_days=("observed_days", "min"),
        maximum_days=("observed_days", "max"),
    )
    .reset_index()
)

monthly_coverage

,month_key,page_rows,clients,minimum_days,maximum_days
0,2026-02,153559,46,1,28
1,2026-03,176738,47,1,31
2,2026-04,194760,51,1,30
3,2026-05,237910,56,1,31
4,2026-06,208636,55,1,30


In [6]:
monthly_cache_path = "/content/flyrank_monthly_summary.pkl"

monthly_summary.to_pickle(monthly_cache_path)

print(f"Temporary cache saved: {monthly_cache_path}")

Temporary cache saved: /content/flyrank_monthly_summary.pkl


In [7]:
monthly_value_columns = [
    "observed_days",
    "impressions",
    "clicks",
    "sum_position",
    "daily_impression_mean",
    "daily_impression_std",
    "daily_position_std",
    "days_with_impressions",
]


def select_month(month, prefix):
    month_frame = monthly_summary.loc[
        monthly_summary["month_key"] == month,
        [
            "client_hash_id",
            "content_hash_id",
            *monthly_value_columns,
        ],
    ].copy()

    rename_map = {
        column: f"{prefix}_{column}"
        for column in monthly_value_columns
    }

    return month_frame.rename(columns=rename_map)


february = select_month("2026-02", "feb")
march = select_month("2026-03", "mar")
april = select_month("2026-04", "apr")
may = select_month("2026-05", "may")

In [8]:
development = (
    february
    .merge(
        march,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        may,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
)

development = development[
    (development["feb_observed_days"] >= 20)
    & (development["mar_observed_days"] >= 20)
    & (development["apr_observed_days"] >= 20)
    & (development["may_observed_days"] >= 20)
    & (development["mar_impressions"] > 0)
    & (development["apr_impressions"] > 0)
].copy()

development = development.reset_index(drop=True)

print(f"Development rows: {len(development):,}")
print(
    "Development clients:",
    development["client_hash_id"].nunique(),
)
print(
    "Duplicate client-page rows:",
    development.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum(),
)

Development rows: 62,558
Development clients: 23
Duplicate client-page rows: 0


In [9]:
# Three-month traffic totals: February through April
development["impressions_90d"] = (
    development["feb_impressions"]
    + development["mar_impressions"]
    + development["apr_impressions"]
)

development["clicks_90d"] = (
    development["feb_clicks"]
    + development["mar_clicks"]
    + development["apr_clicks"]
)

# Log versions reduce the effect of extremely large traffic values.
development["log_impressions_90d"] = np.log1p(
    development["impressions_90d"]
)

development["log_previous_month_impressions"] = np.log1p(
    development["mar_impressions"]
)

development["log_current_month_impressions"] = np.log1p(
    development["apr_impressions"]
)

# March-to-April impression momentum
development["recent_change_pct"] = (
    100
    * (
        development["apr_impressions"]
        - development["mar_impressions"]
    )
    / development["mar_impressions"]
)

# Limit extreme growth percentages caused by small denominators.
development["recent_change_pct_clipped"] = (
    development["recent_change_pct"]
    .clip(lower=-100, upper=500)
)

# Monthly CTR
development["previous_ctr"] = (
    100
    * development["mar_clicks"]
    / development["mar_impressions"]
)

development["current_ctr"] = (
    100
    * development["apr_clicks"]
    / development["apr_impressions"]
)

development["ctr_change"] = (
    development["current_ctr"]
    - development["previous_ctr"]
)

# Impression-weighted average position
development["previous_avg_position"] = (
    development["mar_sum_position"]
    / development["mar_impressions"]
)

development["current_avg_position"] = (
    development["apr_sum_position"]
    / development["apr_impressions"]
)

# Positive means the page's average position became worse.
development["position_change"] = (
    development["current_avg_position"]
    - development["previous_avg_position"]
)

# Current-month traffic volatility
development["current_impression_cv"] = (
    development["apr_daily_impression_std"]
    / development["apr_daily_impression_mean"]
)

development["current_position_std"] = (
    development["apr_daily_position_std"]
)

development["current_impression_day_rate"] = (
    development["apr_days_with_impressions"]
    / development["apr_observed_days"]
)

# May outcome: validation only, never a feature
development["future_change_pct"] = (
    100
    * (
        development["may_impressions"]
        - development["apr_impressions"]
    )
    / development["apr_impressions"]
)

development["future_decline"] = (
    development["may_impressions"]
    < 0.80 * development["apr_impressions"]
).astype(int)

In [10]:
print(
    "Future-decline base rate:",
    f"{development['future_decline'].mean():.2%}",
)

print(
    "Infinite feature values:",
    np.isinf(
        development.select_dtypes(include="number")
    ).sum().sum(),
)

print(
    "Missing current position volatility:",
    development["current_position_std"].isna().sum(),
)

Future-decline base rate: 48.79%
Infinite feature values: 0
Missing current position volatility: 0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped development split because pages belonging to the same client may share traffic patterns, content strategies, and measurement conditions. A random page split could place pages from the same client in both training and validation data, making performance appear stronger than it would be for an unfamiliar client.

I hold out approximately 20% of clients using GroupShuffleSplit with random_state=42. Client identifiers are used only to form the split and are not model features. All models and the Week-4 baseline will be evaluated on the same validation rows.

May is the development outcome month. After model and feature decisions are frozen, I will shift the feature construction forward by one month and use June as a separate temporal test.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

target_column = "future_decline"
group_column = "client_hash_id"

y = development[target_column].astype(int)
groups = development[group_column].astype(str)

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_indices, validation_indices = next(
    group_splitter.split(
        development,
        y,
        groups=groups,
    )
)

train_data = development.iloc[train_indices].copy()
validation_data = development.iloc[validation_indices].copy()

train_clients = set(train_data["client_hash_id"])
validation_clients = set(validation_data["client_hash_id"])

client_overlap = train_clients.intersection(
    validation_clients
)

assert len(client_overlap) == 0
assert train_data[target_column].nunique() == 2
assert validation_data[target_column].nunique() == 2

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_data),
            "clients": train_data["client_hash_id"].nunique(),
            "declining_pages": train_data[target_column].sum(),
            "decline_base_rate": train_data[target_column].mean(),
        },
        {
            "split": "validation",
            "rows": len(validation_data),
            "clients": validation_data[
                "client_hash_id"
            ].nunique(),
            "declining_pages": validation_data[
                target_column
            ].sum(),
            "decline_base_rate": validation_data[
                target_column
            ].mean(),
        },
    ]
)

split_summary["decline_base_rate"] = (
    100 * split_summary["decline_base_rate"]
).round(2)

print(f"Client overlap: {len(client_overlap)}")
split_summary


Client overlap: 0


,split,rows,clients,declining_pages,decline_base_rate
0,train,48915,18,22987,46.99
1,validation,13643,5,7534,55.22


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_columns = [
    # Traffic level
    "log_impressions_90d",
    "log_previous_month_impressions",
    "log_current_month_impressions",

    # Recent traffic direction
    "recent_change_pct_clipped",

    # CTR level and movement
    "previous_ctr",
    "current_ctr",
    "ctr_change",

    # Search position level and movement
    "previous_avg_position",
    "current_avg_position",
    "position_change",

    # Current-month stability
    "current_impression_cv",
    "current_position_std",
    "current_impression_day_rate",
]

forbidden_features = {
    "client_hash_id",
    "content_hash_id",
    "may_impressions",
    "future_change_pct",
    "future_decline",
}

leaked_features = forbidden_features.intersection(
    feature_columns
)

assert len(leaked_features) == 0

X_train = train_data[feature_columns].copy()
y_train = train_data["future_decline"].astype(int)

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["future_decline"].astype(int)

print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Model features: {len(feature_columns)}")
print(f"Forbidden model features: {sorted(leaked_features)}")



Training rows: 48,915
Validation rows: 13,643
Model features: 13
Forbidden model features: []


In [15]:
logistic_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_forest_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                min_samples_leaf=25,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

In [16]:
baseline_qualifies = (
    (validation_data["mar_impressions"] >= 300)
    & (validation_data["recent_change_pct"] < -20)
)

baseline_validation_scores = np.where(
    baseline_qualifies,
    (
        validation_data["mar_impressions"]
        - validation_data["apr_impressions"]
    ),
    0,
)

In [17]:
def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    order = np.argsort(-score_array)
    selected_indices = order[:min(k, len(order))]

    return y_array[selected_indices].mean()


def evaluate_ranking(name, y_true, scores):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    return {
        "method": name,
        "base_rate": y_array.mean(),
        "precision_at_20": precision_at_k(
            y_array, score_array, 20
        ),
        "precision_at_50": precision_at_k(
            y_array, score_array, 50
        ),
        "precision_at_100": precision_at_k(
            y_array, score_array, 100
        ),
        "average_precision": average_precision_score(
            y_array, score_array
        ),
        "roc_auc": roc_auc_score(
            y_array, score_array
        ),
    }

In [18]:
logistic_model.fit(X_train, y_train)

logistic_validation_scores = (
    logistic_model.predict_proba(X_validation)[:, 1]
)

random_forest_model.fit(X_train, y_train)

random_forest_validation_scores = (
    random_forest_model.predict_proba(
        X_validation
    )[:, 1]
)

print("Both models trained successfully.")

Both models trained successfully.


,method,base_rate,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc
0,Week-4 baseline,55.22,70.0,82.0,78.0,64.53,62.67
1,Logistic Regression,55.22,95.0,80.0,82.0,73.97,70.48
2,Random Forest,55.22,100.0,98.0,99.0,81.86,77.72


In [21]:
# ============================================================
# LEAKAGE AUDIT
# ============================================================

model_features = [str(column) for column in X_train.columns]

print("Features supplied to the models:")
for number, feature in enumerate(model_features, start=1):
    print(f"{number:>2}. {feature}")

# These columns must never be model inputs.
forbidden_exact_names = {
    # Outcome and label columns
    "may_impressions",
    "may_days",
    "future_change_pct",
    "future_decline",
    "target",
    "label",

    # Page and client identifiers
    "client_hash_id",
    "content_hash_id",
    "keyword_hash_id",
    "url_hash_id",

    # Product-generated decisions
    "health_score",
    "priority_score",
    "action_type",
    "action_label",
    "reason_code",
    "refresh_flag",

    # This reveals information after the feature-window cutoff
    "updated_after_feature_window",
}

# These patterns catch differently named future or product columns.
forbidden_name_patterns = (
    "may_",
    "_may",
    "future_",
    "_future",
    "target_",
    "_target",
    "label_",
    "_label",
    "_flag",
    "flag_",
)

normalized_features = {
    feature: feature.lower().strip()
    for feature in model_features
}

exact_name_leaks = [
    feature
    for feature, normalized in normalized_features.items()
    if normalized in forbidden_exact_names
]

pattern_leaks = [
    feature
    for feature, normalized in normalized_features.items()
    if any(pattern in normalized for pattern in forbidden_name_patterns)
]

suspected_leaks = sorted(set(exact_name_leaks + pattern_leaks))

# Confirm train and validation use identical feature definitions.
train_only_columns = sorted(
    set(X_train.columns) - set(X_validation.columns)
)
validation_only_columns = sorted(
    set(X_validation.columns) - set(X_train.columns)
)

# Confirm the grouped split still has no overlapping clients.
train_clients = set(train_data["client_hash_id"])
validation_clients = set(validation_data["client_hash_id"])
client_overlap = train_clients.intersection(validation_clients)

# Look for a feature that is an exact copy of the target.
target_copy_columns = []

train_target = pd.Series(y_train).reset_index(drop=True)

for feature in model_features:
    feature_values = pd.to_numeric(
        X_train[feature], errors="coerce"
    ).reset_index(drop=True)

    comparable = feature_values.notna() & train_target.notna()

    if (
        comparable.any()
        and feature_values[comparable].equals(
            train_target[comparable].astype(feature_values.dtype)
        )
    ):
        target_copy_columns.append(feature)

# Produce an easy-to-read audit summary.
leakage_audit = {
    "feature_window_end": "2026-04-30",
    "outcome_window": "2026-05-01 to 2026-05-31",
    "number_of_model_features": len(model_features),
    "forbidden_feature_names_found": suspected_leaks,
    "target_copy_columns_found": target_copy_columns,
    "train_only_columns": train_only_columns,
    "validation_only_columns": validation_only_columns,
    "overlapping_clients": len(client_overlap),
}

print("\nLeakage audit:")
for check, result in leakage_audit.items():
    print(f"{check}: {result}")

# Stop the notebook if any structural leakage is detected.
assert not suspected_leaks, (
    f"Possible future, ID, label, or product leakage: {suspected_leaks}"
)

assert not target_copy_columns, (
    f"These features exactly reproduce the target: {target_copy_columns}"
)

assert not train_only_columns, (
    f"Columns found only in training data: {train_only_columns}"
)

assert not validation_only_columns, (
    f"Columns found only in validation data: {validation_only_columns}"
)

assert len(client_overlap) == 0, (
    f"{len(client_overlap)} clients occur in both splits."
)

print("\nPASS: No structural leakage was detected.")
print("All model inputs are available by the April 30 cutoff.")
print("May information is used only to construct the outcome.")

Features supplied to the models:
 1. log_impressions_90d
 2. log_previous_month_impressions
 3. log_current_month_impressions
 4. recent_change_pct_clipped
 5. previous_ctr
 6. current_ctr
 7. ctr_change
 8. previous_avg_position
 9. current_avg_position
10. position_change
11. current_impression_cv
12. current_position_std
13. current_impression_day_rate

Leakage audit:
feature_window_end: 2026-04-30
outcome_window: 2026-05-01 to 2026-05-31
number_of_model_features: 13
forbidden_feature_names_found: []
target_copy_columns_found: []
train_only_columns: []
validation_only_columns: []
overlapping_clients: 0

PASS: No structural leakage was detected.
All model inputs are available by the April 30 cutoff.
May information is used only to construct the outcome.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

rf_model = random_forest_model

rf_permutation = permutation_importance(
    estimator=rf_model,
    X=X_validation,
    y=y_validation,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

feature_importance = (
    pd.DataFrame({
        "feature": X_validation.columns,
        "importance_mean": rf_permutation.importances_mean * 100,
        "importance_std": rf_permutation.importances_std * 100,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance)

,feature,importance_mean,importance_std
0,current_impression_cv,6.945754,0.141740
1,current_avg_position,5.107035,0.278505
2,current_position_std,4.028216,0.283061
3,log_current_month_impressions,3.845948,0.166990
4,previous_avg_position,0.889634,0.080924
5,log_previous_month_impressions,0.687845,0.141699
6,current_ctr,0.377909,0.065216
7,recent_change_pct_clipped,0.212832,0.035574
8,previous_ctr,0.159650,0.026239
9,position_change,0.109516,0.038140


The Random Forest relied most strongly on April impression volatility, April average search position, April position volatility, and April impression volume. The Week-4 baseline relied primarily on March-to-April impression decline, but that feature had relatively low incremental importance in the Random Forest. This suggests that current instability and search-ranking behavior contain predictive information beyond the baseline trend rule. These relationships are predictive rather than causal, and position-related signals cannot distinguish among competition, search-demand changes, algorithm changes, and changes in content relevance.

In [25]:
# ============================================================
# RANDOM FOREST ERROR REVIEW AT K = 100
# ============================================================

REVIEW_CAPACITY = 100

# Retain useful page information that exists in validation_data.
desired_context_columns = [
    "client_hash_id",
    "content_hash_id",
    "mar_impressions",
    "apr_impressions",
    "may_impressions",
    "recent_change_pct",
    "future_change_pct",
]

context_columns = [
    column
    for column in desired_context_columns
    if column in validation_data.columns
]

error_review = (
    validation_data[context_columns]
    .reset_index(drop=True)
    .copy()
)

# Add the real May outcome and the Random Forest probability.
error_review["actual_future_decline"] = (
    np.asarray(y_validation).astype(int)
)

error_review["rf_decline_probability"] = (
    random_forest_model.predict_proba(X_validation)[:, 1]
)

# Rank pages from highest to lowest predicted decline risk.
error_review["model_rank"] = (
    error_review["rf_decline_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

error_review["selected_for_review"] = (
    error_review["model_rank"] <= REVIEW_CAPACITY
)

# Describe each page relative to the top-100 review cutoff.
error_review["review_result"] = np.select(
    [
        (
            error_review["selected_for_review"]
            & (error_review["actual_future_decline"] == 1)
        ),
        (
            error_review["selected_for_review"]
            & (error_review["actual_future_decline"] == 0)
        ),
        (
            ~error_review["selected_for_review"]
            & (error_review["actual_future_decline"] == 1)
        ),
    ],
    [
        "correctly_selected_decline",
        "false_positive",
        "decline_outside_top_100",
    ],
    default="correctly_not_selected",
)

error_summary = (
    error_review["review_result"]
    .value_counts()
    .rename_axis("review_result")
    .reset_index(name="n")
)

display(error_summary)

print("\nFalse positives in the top 100:")
display(
    error_review.loc[
        error_review["review_result"] == "false_positive"
    ]
    .sort_values("model_rank")
)

print("\nHighest-ranked declining pages outside the top 100:")
display(
    error_review.loc[
        error_review["review_result"] == "decline_outside_top_100"
    ]
    .sort_values("model_rank")
    .head(10)
)

,review_result,n
0,decline_outside_top_100,7435
1,correctly_not_selected,6108
2,correctly_selected_decline,99
3,false_positive,1



False positives in the top 100:


,client_hash_id,content_hash_id,mar_impressions,apr_impressions,may_impressions,recent_change_pct,future_change_pct,actual_future_decline,rf_decline_probability,model_rank,selected_for_review,review_result
12947,client_62f4a7e64f5e0096,content_f600793b50c1c6f6,3431.0,1499.0,1313.0,-56.310114,-12.408272,0,0.943108,39,True,false_positive



Highest-ranked declining pages outside the top 100:


,client_hash_id,content_hash_id,mar_impressions,apr_impressions,may_impressions,recent_change_pct,future_change_pct,actual_future_decline,rf_decline_probability,model_rank,selected_for_review,review_result
10878,client_62f4a7e64f5e0096,content_cd3930174f56f7c9,2062.0,1912.0,271.0,-7.274491,-85.826360,1,0.930320,101,False,decline_outside_top_100
11037,client_62f4a7e64f5e0096,content_d092a737bb522adf,3003.0,2849.0,509.0,-5.128205,-82.134082,1,0.929852,102,False,decline_outside_top_100
9171,client_62f4a7e64f5e0096,content_acf2fb81d02a8656,647.0,565.0,403.0,-12.673879,-28.672566,1,0.929132,103,False,decline_outside_top_100
3072,client_62f4a7e64f5e0096,content_367b5a72c90bc413,4131.0,1662.0,367.0,-59.767611,-77.918171,1,0.929078,104,False,decline_outside_top_100
11597,client_62f4a7e64f5e0096,content_db6b3a532ae663a1,4375.0,3536.0,148.0,-19.177143,-95.814480,1,0.928996,105,False,decline_outside_top_100
1713,client_62f4a7e64f5e0096,content_1cc10e240de2e502,2485.0,1235.0,447.0,-50.301811,-63.805668,1,0.928831,107,False,decline_outside_top_100
12999,client_62f4a7e64f5e0096,content_f6ecc33354497c23,1224.0,842.0,156.0,-31.209150,-81.472684,1,0.928812,108,False,decline_outside_top_100
11394,client_62f4a7e64f5e0096,content_d7c6539a96cb16aa,1092.0,465.0,116.0,-57.417582,-75.053763,1,0.928754,109,False,decline_outside_top_100
1835,client_62f4a7e64f5e0096,content_1f0e826ab40b4565,1818.0,867.0,327.0,-52.310231,-62.283737,1,0.928667,110,False,decline_outside_top_100
4038,client_62f4a7e64f5e0096,content_49dacf74d8fa83de,2798.0,3143.0,348.0,12.330236,-88.927776,1,0.928160,111,False,decline_outside_top_100


In [26]:
# ============================================================
# CHECK WHETHER ONE CLIENT DOMINATES THE TOP-100 QUEUE
# ============================================================

client_queue_summary = (
    error_review
    .groupby("client_hash_id")
    .agg(
        validation_pages=("actual_future_decline", "size"),
        actual_declining_pages=("actual_future_decline", "sum"),
        average_model_score=("rf_decline_probability", "mean"),
        selected_in_top_100=("selected_for_review", "sum"),
    )
    .reset_index()
)

client_queue_summary["client_base_rate_pct"] = (
    100
    * client_queue_summary["actual_declining_pages"]
    / client_queue_summary["validation_pages"]
)

correctly_selected_by_client = (
    error_review.loc[error_review["selected_for_review"]]
    .groupby("client_hash_id")["actual_future_decline"]
    .sum()
)

client_queue_summary["correct_selected"] = (
    client_queue_summary["client_hash_id"]
    .map(correctly_selected_by_client)
    .fillna(0)
    .astype(int)
)

client_queue_summary["selected_precision_pct"] = np.where(
    client_queue_summary["selected_in_top_100"] > 0,
    (
        100
        * client_queue_summary["correct_selected"]
        / client_queue_summary["selected_in_top_100"]
    ),
    np.nan,
)

client_queue_summary = client_queue_summary.sort_values(
    "selected_in_top_100",
    ascending=False,
)

display(client_queue_summary)

,client_hash_id,validation_pages,actual_declining_pages,average_model_score,selected_in_top_100,client_base_rate_pct,correct_selected,selected_precision_pct
2,client_62f4a7e64f5e0096,13184,7391,0.502310,100,56.060376,99,99.0
0,client_0797ff3a1fc9a6a5,8,0,0.507494,0,0.000000,0,NaN
1,client_400c21c81c8b46ef,250,77,0.424781,0,30.800000,0,NaN
3,client_b10cb2997d0c7c86,162,41,0.319427,0,25.308642,0,NaN
4,client_cd12bcfd98942aa1,39,25,0.408218,0,64.102564,0,NaN


The validation set was strongly unbalanced across clients. One client contributed 13,184 of 13,643 validation pages (96.64%) and all 100 pages in the global top-100 queue. Therefore, the 99% Precision@100 result is valid as a page-weighted global ranking result but primarily reflects performance on this large client. Client identifiers were excluded, so this is not direct identifier leakage. Nevertheless, the result does not demonstrate equally strong performance across all held-out clients. Within-client metrics are reported as a robustness check.

In [27]:
from sklearn.metrics import average_precision_score, roc_auc_score

# Reconstruct the Week-4 baseline score using only March-April data.
error_review["baseline_score"] = np.where(
    (
        (error_review["mar_impressions"] >= 300)
        & (error_review["recent_change_pct"] < -20)
    ),
    error_review["mar_impressions"]
    - error_review["apr_impressions"],
    0,
)

def calculate_client_metrics(
    data,
    score_column,
    method_name,
    requested_k=20,
):
    results = []

    for client_id, client_data in data.groupby("client_hash_id"):
        client_data = client_data.copy()

        y_true = client_data["actual_future_decline"].to_numpy()
        scores = client_data[score_column].to_numpy()

        # Small clients may contain fewer than 20 pages.
        k_used = min(requested_k, len(client_data))

        highest_ranked_indices = np.argsort(-scores)[:k_used]
        precision_at_k = y_true[highest_ranked_indices].mean()

        # Average precision needs at least one declining page.
        if y_true.sum() > 0:
            average_precision = average_precision_score(
                y_true,
                scores,
            )
        else:
            average_precision = np.nan

        # ROC AUC requires both outcome classes.
        if len(np.unique(y_true)) == 2:
            roc_auc = roc_auc_score(y_true, scores)
        else:
            roc_auc = np.nan

        results.append({
            "client_hash_id": client_id,
            "method": method_name,
            "pages": len(client_data),
            "declining_pages": int(y_true.sum()),
            "base_rate_pct": 100 * y_true.mean(),
            "k_used": k_used,
            "precision_at_k_pct": 100 * precision_at_k,
            "average_precision_pct": 100 * average_precision,
            "roc_auc_pct": 100 * roc_auc,
        })

    return pd.DataFrame(results)


baseline_client_metrics = calculate_client_metrics(
    data=error_review,
    score_column="baseline_score",
    method_name="Week-4 baseline",
)

rf_client_metrics = calculate_client_metrics(
    data=error_review,
    score_column="rf_decline_probability",
    method_name="Random Forest",
)

client_metric_comparison = pd.concat(
    [baseline_client_metrics, rf_client_metrics],
    ignore_index=True,
).sort_values(
    ["client_hash_id", "method"]
)

display(client_metric_comparison.round(2))

,client_hash_id,method,pages,declining_pages,base_rate_pct,k_used,precision_at_k_pct,average_precision_pct,roc_auc_pct
5,client_0797ff3a1fc9a6a5,Random Forest,8,0,0.00,8,0.0,NaN,NaN
0,client_0797ff3a1fc9a6a5,Week-4 baseline,8,0,0.00,8,0.0,NaN,NaN
6,client_400c21c81c8b46ef,Random Forest,250,77,30.80,20,50.0,41.36,60.10
1,client_400c21c81c8b46ef,Week-4 baseline,250,77,30.80,20,35.0,37.81,60.22
7,client_62f4a7e64f5e0096,Random Forest,13184,7391,56.06,20,100.0,82.44,77.97
2,client_62f4a7e64f5e0096,Week-4 baseline,13184,7391,56.06,20,70.0,64.99,62.34
8,client_b10cb2997d0c7c86,Random Forest,162,41,25.31,20,35.0,30.46,56.74
3,client_b10cb2997d0c7c86,Week-4 baseline,162,41,25.31,20,30.0,29.56,52.11
9,client_cd12bcfd98942aa1,Random Forest,39,25,64.10,20,45.0,61.63,38.29
4,client_cd12bcfd98942aa1,Week-4 baseline,39,25,64.10,20,70.0,64.10,46.43


In [28]:
# Exclude clients with no declining pages because their
# ranking metrics cannot be meaningfully evaluated.
evaluable_client_metrics = client_metric_comparison.loc[
    client_metric_comparison["declining_pages"] > 0
].copy()

macro_client_comparison = (
    evaluable_client_metrics
    .groupby("method")
    .agg(
        evaluable_clients=("client_hash_id", "nunique"),
        macro_precision_at_k_pct=("precision_at_k_pct", "mean"),
        macro_average_precision_pct=("average_precision_pct", "mean"),
        macro_roc_auc_pct=("roc_auc_pct", "mean"),
    )
    .reset_index()
)

display(macro_client_comparison.round(2))

,method,evaluable_clients,macro_precision_at_k_pct,macro_average_precision_pct,macro_roc_auc_pct
0,Random Forest,4,57.50,53.97,58.27
1,Week-4 baseline,4,51.25,49.12,55.28


The Random Forest’s most influential features were April impression volatility, April average Google position, April position volatility, and April impression volume. This indicates that the model learned signals beyond the Week-4 baseline’s March-to-April impression-decline rule. These relationships are predictive and should not be interpreted as causal.

In the global top-100 queue, 99 pages declined by more than 20% in May and one did not. The false positive had already lost 56.31% of its impressions from March to April, so its selection was reasonable, but its additional May decline was only 12.41% and therefore did not cross the target’s 20% threshold. Several high-ranked pages outside the top 100 experienced sudden May collapses despite showing little decline—or even growth—from March to April. Such reversals were difficult to anticipate using only information available through April.

The validation clients were strongly unbalanced. One client contributed 13,184 of the 13,643 validation pages and all 100 pages in the global top-100 queue. The Random Forest improved within-client Precision@20 for three of the four clients containing positive outcomes, but it performed poorly on one small 39-page client. For that client, Precision@20 was 45% compared with the 64.10% base rate and 70% for the baseline. This demonstrates that the model does not generalize equally across every client.

The Random Forest is retained as the preferred model because it beats the Week-4 baseline on the primary global ranking metrics and on macro-averaged client metrics. Nevertheless, its excellent global Precision@100 primarily reflects performance on one large client. Future evaluation should use additional grouped folds, more balanced client samples, or a later temporal test before deployment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.